In [1]:
# CELL 1: Imports

import os, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
from torch.cuda.amp import autocast
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")


PyTorch  : 2.7.1+cu118
Device   : cuda


In [2]:
# CELL 2 FIXED: Correct checkpoint filenames

BASE_DIR   = r"D:\Major Project"
DATA_DIR   = r"D:\Major Project\chest_xray_data"

# ── Confirmed from disk scan ──────────────────────────────────────────────
PATH_B0    = os.path.join(BASE_DIR, "best_balanced_effnet.pth")    # 15.6 MB ✅
PATH_B3    = os.path.join(BASE_DIR, "best_efficientnet_b3.pth")    # 44.4 MB ✅
PATH_VIT   = os.path.join(BASE_DIR, "best_vit_base.pth")           # 328.9 MB ✅
PATH_DINO  = os.path.join(BASE_DIR, "best_ultimate_dinov2.pth")    # 334.2 MB ✅
PATH_CNX   = os.path.join(BASE_DIR, "best_convnext_v2.pth")        # to be created

LABELS = [
    "Atelectasis","Consolidation","Infiltration","Pneumothorax",
    "Edema","Emphysema","Fibrosis","Effusion","Pneumonia",
    "PleuralThickening","Cardiomegaly","Nodule","Mass","Hernia"
]
NUM_CLASSES  = len(LABELS)
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 4
LABEL_SMOOTH = 0.05

minority_classes = ["Hernia","Pneumonia","Edema","Emphysema","Fibrosis"]

# ── Verify all Phase 7 checkpoints exist ─────────────────────────────────
print("Verifying checkpoint paths:\n")
for name, path in [("B0",    PATH_B0),
                   ("B3",    PATH_B3),
                   ("ViT",   PATH_VIT),
                   ("DINOv2",PATH_DINO)]:
    exists  = os.path.exists(path)
    size_mb = os.path.getsize(path)/1024**2 if exists else 0
    status  = f"✅  {size_mb:.1f} MB" if exists else "❌  NOT FOUND"
    print(f"  {name:<8} {status:<20} {os.path.basename(path)}")

print(f"\nDevice: {DEVICE}")


Verifying checkpoint paths:

  B0       ✅  15.6 MB           best_balanced_effnet.pth
  B3       ✅  44.4 MB           best_efficientnet_b3.pth
  ViT      ✅  328.9 MB          best_vit_base.pth
  DINOv2   ✅  334.2 MB          best_ultimate_dinov2.pth

Device: cuda


In [3]:
# CELL 3: Load all saved predictions from Phase 7 — no recomputation needed

def load_cache(name):
    path = os.path.join(BASE_DIR, f"cache_{name}.npy")
    assert os.path.exists(path), f"Missing: {path} — re-run Phase 7 resume cell"
    arr = np.load(path)
    print(f"  ✅ {name:<20} shape: {arr.shape}")
    return arr

print("Loading Phase 7 cached predictions...\n")
test_labels = load_cache("test_labels")
b0_preds    = load_cache("b0_preds")
b3_preds    = load_cache("b3_preds")
vit_preds   = load_cache("vit_preds")
dino_preds  = load_cache("dino_preds")
print(f"\nTest samples: {len(test_labels)}")


Loading Phase 7 cached predictions...

  ✅ test_labels          shape: (25596, 14)
  ✅ b0_preds             shape: (25596, 14)
  ✅ b3_preds             shape: (25596, 14)
  ✅ vit_preds            shape: (25596, 14)
  ✅ dino_preds           shape: (25596, 14)

Test samples: 25596


In [4]:
# CELL 4: Safe AUC helper + print Phase 7 baseline

def safe_roc_auc(y_true, y_pred):
    n       = y_true.shape[1]
    pc      = np.full(n, np.nan)
    for i in range(n):
        if len(np.unique(y_true[:, i])) < 2:
            continue
        pc[i] = roc_auc_score(y_true[:, i], y_pred[:, i])
    return float(np.nanmean(pc)), pc


# Recompute Phase 7 best ensemble as baseline
wb0, wb3, wvit, wdino = 0.25, 0.15, 0.30, 0.30
baseline_preds = wb0*b0_preds + wb3*b3_preds + wvit*vit_preds + wdino*dino_preds
baseline_auc, baseline_pc = safe_roc_auc(test_labels, baseline_preds)

print(f"Phase 7 Baseline (4-Way Ensemble) AUC: {baseline_auc:.4f}")
print(f"\n  {'Disease':<22} {'Baseline':>10}")
print(f"  {'-'*36}")
for l, a in zip(LABELS, baseline_pc):
    val = f"{a:.4f}" if not np.isnan(a) else "  N/A "
    flag = "  ⚠️ " if (not np.isnan(a) and a < 0.75) else ""
    print(f"  {l:<22} {val:>10}{flag}")


Phase 7 Baseline (4-Way Ensemble) AUC: 0.8592

  Disease                  Baseline
  ------------------------------------
  Atelectasis                0.7533
  Consolidation              0.7703
  Infiltration               0.7189  ⚠️ 
  Pneumothorax               0.9002
  Edema                      0.9231
  Emphysema                  0.9570
  Fibrosis                   0.9424
  Effusion                   0.8159
  Pneumonia                  0.8963
  PleuralThickening            N/A 
  Cardiomegaly               0.9323
  Nodule                     0.7531
  Mass                       0.8187
  Hernia                     0.9885


In [5]:
# CELL 4c-PATCH: Rebuild all loaders with num_workers=0

train_ds = ChestXrayDataset(df_train, IMAGE_MAP, get_transforms(train=True))
val_ds   = ChestXrayDataset(df_val,   IMAGE_MAP, get_transforms(train=False))
test_ds  = ChestXrayDataset(df_test,  IMAGE_MAP, get_transforms(train=False))

TRAIN_DL = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, pin_memory=False)   # ✅ Windows fix
VAL_DL   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                      num_workers=0, pin_memory=False)
TEST_DL  = DataLoader(test_ds,  batch_size=64, shuffle=False,
                      num_workers=0, pin_memory=False)

TRAIN_DL_CNX = TRAIN_DL
VAL_DL_CNX   = VAL_DL
TEST_DL_CNX  = TEST_DL

print(f"  Train batches : {len(TRAIN_DL):,}")
print(f"  Val batches   : {len(VAL_DL):,}")
print(f"  Test batches  : {len(TEST_DL):,}")
print("✅ Loaders rebuilt — num_workers=0")


Scanning: D:\Major Project

Found 11 checkpoint(s):

      15.6 MB  →  D:\Major Project\best_balanced_effnet.pth
      44.4 MB  →  D:\Major Project\best_efficientnet_b3.pth
     334.2 MB  →  D:\Major Project\best_ultimate_dinov2.pth
     328.9 MB  →  D:\Major Project\best_vit_base.pth
      46.5 MB  →  D:\Major Project\experiments\chest_xray_20251022_093109\best_model.pth
      15.6 MB  →  D:\Major Project\experiments\chest_xray_20251022_093109\latest.pth
      46.5 MB  →  D:\Major Project\experiments\chest_xray_224x224_20251023_101202\best_model.pth
      46.5 MB  →  D:\Major Project\experiments\chest_xray_FULL_dataset_20251021_230048\best_model.pth
      15.6 MB  →  D:\Major Project\experiments\chest_xray_FULL_dataset_20251021_230048\latest.pth
      46.5 MB  →  D:\Major Project\experiments\chest_xray_efficientnet_b0_20251021_195318\best_model.pth
      15.6 MB  →  D:\Major Project\experiments\chest_xray_efficientnet_b0_20251021_195318\latest.pth


In [6]:
# CELL 4b: Load all Phase 7 models into memory
# Run this BEFORE Cell 5 (TTA) and BEFORE Cell 10 (ensemble)

from efficientnet_pytorch import EfficientNet
import timm
import torch.nn as nn

NUM_CLASSES = 14   # confirm this matches your setup

# ── Paste your exact class definitions from old notebook ─────────────────

class EfficientNetB0(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.net = EfficientNet.from_pretrained("efficientnet-b0")
        in_features = self.net._fc.in_features
        self.net._fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )
    def forward(self, x, metadata=None):
        return self.net(x)


class EfficientNetB3(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.4):
        super().__init__()
        self.net = EfficientNet.from_pretrained("efficientnet-b3")
        in_features = self.net._fc.in_features
        self.net._fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )
    def forward(self, x, metadata=None):
        return self.net(x)


class ViTBase(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.vit = timm.create_model(
            "vit_base_patch16_224", pretrained=False, num_classes=0
        )
        feat_dim = self.vit.num_features  # 768
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim),     # 0
            nn.Dropout(dropout),        # 1
            nn.Linear(feat_dim, 512),   # 2
            nn.GELU(),                  # 3
            nn.Dropout(dropout * 0.67), # 4
            nn.Linear(512, num_classes) # 5
        )
    def forward(self, x, metadata=None):
        return self.classifier(self.vit(x))


class AnatomicalAttention(nn.Module):
    def __init__(self, feat_dim=768):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(feat_dim, feat_dim//4), nn.ReLU(),
            nn.Linear(feat_dim//4, feat_dim), nn.Sigmoid(),
        )
    def forward(self, spatial_feats):
        gap = spatial_feats.mean(dim=[2, 3])
        return gap * self.gate(gap)


class DINOv2MultiModal(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, freeze_blocks=8, dropout=0.3):
        super().__init__()
        self.backbone = torch.hub.load(
            "facebookresearch/dinov2", "dinov2_vitb14", pretrained=True
        )
        for p in self.backbone.parameters():
            p.requires_grad = False
        for i in range(freeze_blocks, 12):
            for p in self.backbone.blocks[i].parameters():
                p.requires_grad = True
        for p in self.backbone.norm.parameters():
            p.requires_grad = True
        feat_dim = self.backbone.embed_dim  # 768
        self.attn = AnatomicalAttention(feat_dim)
        self.meta_encoder = nn.Sequential(
            nn.Linear(4, 64),    nn.ReLU(),
            nn.Linear(64, 128),  nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
        )
        self.fusion_head = nn.Sequential(
            nn.LayerNorm(feat_dim + 256),
            nn.Linear(feat_dim + 256, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),            nn.GELU(), nn.Dropout(dropout*0.67),
            nn.Linear(256, num_classes),
        )
    def forward(self, images, metadata):
        B   = images.shape[0]
        out = self.backbone.forward_features(images)
        patch_tokens = out["x_norm_patchtokens"]
        H = W        = int(patch_tokens.shape[1] ** 0.5)
        spatial      = patch_tokens.permute(0,2,1).reshape(B,-1,H,W)
        cls_token    = out["x_norm_clstoken"]
        img_feat     = self.attn(spatial) + cls_token
        meta_feat    = self.meta_encoder(metadata)
        return self.fusion_head(torch.cat([img_feat, meta_feat], dim=1))


# ── Load weights from Phase 7 checkpoints ────────────────────────────────
print("Loading Phase 7 model weights...\n")

model_b0 = EfficientNetB0().to(DEVICE)
model_b0.load_state_dict(torch.load(PATH_B0, map_location=DEVICE))
model_b0.eval()
print("  ✅ EfficientNet-B0 loaded")

model_b3 = EfficientNetB3().to(DEVICE)
model_b3.load_state_dict(torch.load(PATH_B3, map_location=DEVICE))
model_b3.eval()
print("  ✅ EfficientNet-B3 loaded")

model_vit = ViTBase().to(DEVICE)
model_vit.load_state_dict(torch.load(PATH_VIT, map_location=DEVICE))
model_vit.eval()
print("  ✅ ViT-Base/16 loaded")

model_dino = DINOv2MultiModal(freeze_blocks=8).to(DEVICE)
model_dino.load_state_dict(torch.load(PATH_DINO, map_location=DEVICE))
model_dino.eval()
print("  ✅ DINOv2 MultiModal loaded")

print("\n✅ All 4 models ready.")


Loading Phase 7 model weights...

Loaded pretrained weights for efficientnet-b0
  ✅ EfficientNet-B0 loaded
Loaded pretrained weights for efficientnet-b3
  ✅ EfficientNet-B3 loaded
  ✅ ViT-Base/16 loaded


Using cache found in C:\Users\91850/.cache\torch\hub\facebookresearch_dinov2_main


  ✅ DINOv2 MultiModal loaded

✅ All 4 models ready.


In [1]:
# CELL 4c FINAL: Official NIH train/val/test split
# CELL 4c-PART1: Build image map first (run this before Part 2)

import os
from pathlib import Path

def build_image_map(data_dir):
    img_map = {}
    for root, dirs, files in os.walk(data_dir):
        for fname in files:
            if fname.lower().endswith(".png"):
                img_map[fname] = os.path.join(root, fname)
    return img_map

print("Scanning for images...")
IMAGE_MAP = build_image_map(DATA_DIR)
print(f"✅ Images found: {len(IMAGE_MAP):,}")

import pandas as pd
import numpy as np
import torch
import os
from torch.utils.data import Dataset, DataLoader
from PIL import Image as PILImage
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

# ── Load official split file lists ───────────────────────────────────────
with open(os.path.join(DATA_DIR, "test_list.txt")) as f:
    test_images = set(line.strip() for line in f.readlines())

with open(os.path.join(DATA_DIR, "train_val_list.txt")) as f:
    train_val_images = set(line.strip() for line in f.readlines())

print(f"Official test images     : {len(test_images):,}")
print(f"Official train+val images: {len(train_val_images):,}")
print(f"Total                    : {len(test_images)+len(train_val_images):,}")

# ── Load + parse master CSV ───────────────────────────────────────────────
MASTER_CSV = os.path.join(DATA_DIR, "Data_Entry_2017.csv")
df_all = pd.read_csv(MASTER_CSV).rename(columns={"Image Index": "Image_Index"})

def parse_labels(df, labels):
    for lbl in labels:
        df[lbl] = df["Finding Labels"].apply(
            lambda x: 1.0 if lbl in str(x) else 0.0
        )
    return df

df_all = parse_labels(df_all, LABELS)

# ── Apply official splits ─────────────────────────────────────────────────
df_test     = df_all[df_all["Image_Index"].isin(test_images)].reset_index(drop=True)
df_trainval = df_all[df_all["Image_Index"].isin(train_val_images)].reset_index(drop=True)

# Split train+val → 90% train / 10% val (same seed as original)
df_train, df_val = train_test_split(
    df_trainval, test_size=0.1, random_state=42, shuffle=True
)

print(f"\nSplit sizes:")
print(f"  Train : {len(df_train):,}")
print(f"  Val   : {len(df_val):,}")
print(f"  Test  : {len(df_test):,}  ← cache has: {len(test_labels):,}")

assert len(df_test) == len(test_labels), \
    f"❌ Still mismatched: {len(df_test)} vs {len(test_labels)}"
print(f"\n✅ Test split matches cache exactly!")

# ── Save splits ───────────────────────────────────────────────────────────
df_train.to_csv(os.path.join(BASE_DIR, "train_split.csv"), index=False)
df_val.to_csv(os.path.join(BASE_DIR,   "val_split.csv"),   index=False)
df_test.to_csv(os.path.join(BASE_DIR,  "test_split.csv"),  index=False)
print("✅ Splits saved.")

# ── Transforms ────────────────────────────────────────────────────────────
def get_transforms(train=True):
    if train:
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                               rotate_limit=10, p=0.5),
            A.GaussNoise(p=0.2),
            A.CLAHE(p=0.3),
            A.CoarseDropout(max_holes=8, max_height=24,
                            max_width=24, p=0.3),
            A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            ToTensorV2()
        ])
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ToTensorV2()
    ])

# ── Dataset ───────────────────────────────────────────────────────────────
class ChestXrayDataset(Dataset):
    def __init__(self, df, image_map, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_map = image_map
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        fname = row["Image_Index"]
        path  = self.image_map.get(fname, None)

        if path is None or not os.path.exists(path):
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = np.array(PILImage.open(path).convert("RGB"))

        if self.transform:
            img = self.transform(image=img)["image"]

        labels = torch.FloatTensor([row.get(l, 0.0) for l in LABELS])

        age    = np.clip(float(row.get("Patient Age",    50)), 0, 95) / 95.0
        gender = {"M": 0.0, "F": 1.0}.get(
                    str(row.get("Patient Gender", "M")).upper(), 0.5)
        view   = {"PA": 0.0, "AP": 1.0}.get(
                    str(row.get("View Position",  "PA")).upper(), 0.5)
        fup    = np.clip(float(row.get("Follow-up #", 0)) / 10.0, 0, 1)
        meta   = torch.FloatTensor([age, gender, view, fup])

        return img, labels, meta

# CELL 4c-PATCH: Rebuild all loaders with num_workers=0

train_ds = ChestXrayDataset(df_train, IMAGE_MAP, get_transforms(train=True))
val_ds   = ChestXrayDataset(df_val,   IMAGE_MAP, get_transforms(train=False))
test_ds  = ChestXrayDataset(df_test,  IMAGE_MAP, get_transforms(train=False))

TRAIN_DL = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=0, pin_memory=False)   # ✅ Windows fix
VAL_DL   = DataLoader(val_ds,   batch_size=64, shuffle=False,
                      num_workers=0, pin_memory=False)
TEST_DL  = DataLoader(test_ds,  batch_size=64, shuffle=False,
                      num_workers=0, pin_memory=False)

TRAIN_DL_CNX = TRAIN_DL
VAL_DL_CNX   = VAL_DL
TEST_DL_CNX  = TEST_DL

print(f"  Train batches : {len(TRAIN_DL):,}")
print(f"  Val batches   : {len(VAL_DL):,}")
print(f"  Test batches  : {len(TEST_DL):,}")
print("✅ Loaders rebuilt — num_workers=0")


# ── Class weights ──────────────────────────────────────────────────────────
label_counts  = df_train[LABELS].sum()
total         = len(df_train)
CLASS_WEIGHTS = torch.FloatTensor([
    total / (len(LABELS) * max(label_counts[l], 1)) for l in LABELS
]).to(DEVICE)

print(f"\n  Train batches : {len(TRAIN_DL):,}")
print(f"  Val batches   : {len(VAL_DL):,}")
print(f"  Test batches  : {len(TEST_DL):,}")
print(f"\n✅ DataLoaders ready — proceed to Cell 5 (TTA).")


Scanning for images...


NameError: name 'DATA_DIR' is not defined

In [8]:
# CHECK: Official NIH split files
import os

official_test  = os.path.join(DATA_DIR, "test_list.txt")
official_train = os.path.join(DATA_DIR, "train_val_list.txt")

for f in [official_test, official_train]:
    exists = os.path.exists(f)
    if exists:
        with open(f) as fp:
            lines = fp.readlines()
        print(f"✅ {os.path.basename(f):<25} {len(lines):,} entries")
    else:
        print(f"❌ NOT FOUND: {f}")


✅ test_list.txt             25,596 entries
✅ train_val_list.txt        86,524 entries


In [9]:
# CELL 5 FIXED: TTA — Windows-safe, dedicated loader per augmentation

import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

TTA_TRANSFORMS = [
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
               A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
               ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
               A.HorizontalFlip(p=1.0),
               A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
               ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE),
               A.RandomBrightnessContrast(0.1, 0.1, p=1.0),
               A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
               ToTensorV2()]),
]

def make_tta_loader(df, image_map, transform):
    """
    Creates a fresh DataLoader with num_workers=0 for each augmentation.
    This avoids Windows multiprocessing deadlocks when swapping transforms.
    """
    ds = ChestXrayDataset(df, image_map, transform)
    return DataLoader(
        ds,
        batch_size  = 64,
        shuffle     = False,
        num_workers = 0,      # ✅ critical fix for Windows
        pin_memory  = False,
    )


@torch.no_grad()
def run_tta(model, df, image_map, use_meta=False):
    model.eval()
    aug_preds = []

    for aug_idx, transform in enumerate(TTA_TRANSFORMS):
        loader = make_tta_loader(df, image_map, transform)
        preds  = []
        for x, y, meta in tqdm(loader,
                                desc=f"  aug {aug_idx+1}/{len(TTA_TRANSFORMS)}",
                                ncols=100):
            x = x.to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(x, meta.to(DEVICE)) if use_meta \
                         else model(x, None)
            preds.append(torch.sigmoid(logits).cpu().float().numpy())
        aug_preds.append(np.vstack(preds))
        del loader   # release immediately

    return np.mean(aug_preds, axis=0)


def load_or_run_tta(model, df, image_map, name, use_meta=False):
    cache_path = os.path.join(BASE_DIR, f"cache_tta_{name}.npy")
    if os.path.exists(cache_path):
        arr = np.load(cache_path)
        print(f"  ✅ Loaded from cache : tta_{name}  shape={arr.shape}")
        return arr
    print(f"  🔄 Running TTA       : {name}")
    arr = run_tta(model, df, image_map, use_meta=use_meta)
    np.save(cache_path, arr)
    print(f"  💾 Cached            : cache_tta_{name}.npy")
    return arr


# ── Run TTA on all 4 models ───────────────────────────────────────────────
print("Running TTA (3 augmentations, Windows-safe)...\n")

b0_tta   = load_or_run_tta(model_b0,   df_test, IMAGE_MAP, "b0",   use_meta=False)
b3_tta   = load_or_run_tta(model_b3,   df_test, IMAGE_MAP, "b3",   use_meta=False)
vit_tta  = load_or_run_tta(model_vit,  df_test, IMAGE_MAP, "vit",  use_meta=False)
dino_tta = load_or_run_tta(model_dino, df_test, IMAGE_MAP, "dino", use_meta=True)

# ── Ensemble TTA predictions ──────────────────────────────────────────────
wb0, wb3, wvit, wdino = 0.25, 0.15, 0.30, 0.30
tta_ens            = wb0*b0_tta + wb3*b3_tta + wvit*vit_tta + wdino*dino_tta
tta_auc,  tta_pc   = safe_roc_auc(test_labels, tta_ens)

print(f"\n{'='*55}")
print(f"  Baseline (no TTA)   AUC : {baseline_auc:.4f}")
print(f"  TTA Ensemble (3-aug) AUC : {tta_auc:.4f}")
print(f"  Improvement              : +{tta_auc - baseline_auc:.4f}")
print(f"{'='*55}")

print("\nPer-class change:")
for l, a, b in zip(LABELS, tta_pc, baseline_pc):
    if np.isnan(a):
        continue
    d     = a - b if not np.isnan(b) else 0
    arrow = "↑" if d > 0.002 else ("↓" if d < -0.002 else "–")
    flag  = "  ✅" if a >= 0.85 else ("  ⚠️ " if a < 0.75 else "")
    print(f"  {l:<22} {a:.4f}  {arrow} {d:+.4f}{flag}")


Running TTA (3 augmentations, Windows-safe)...

  🔄 Running TTA       : b0


  aug 3/3: 100%|██████████████████████████████████████████████████| 400/400 [24:58<00:00,  3.75s/it]


  💾 Cached            : cache_tta_b0.npy
  🔄 Running TTA       : b3


  aug 3/3: 100%|██████████████████████████████████████████████████| 400/400 [24:40<00:00,  3.70s/it]


  💾 Cached            : cache_tta_b3.npy
  🔄 Running TTA       : vit


  aug 3/3: 100%|██████████████████████████████████████████████████| 400/400 [29:02<00:00,  4.36s/it]


  💾 Cached            : cache_tta_vit.npy
  🔄 Running TTA       : dino


  aug 3/3: 100%|██████████████████████████████████████████████████| 400/400 [29:33<00:00,  4.43s/it]


  💾 Cached            : cache_tta_dino.npy

  Baseline (no TTA)   AUC : 0.8592
  TTA Ensemble (3-aug) AUC : 0.8607
  Improvement              : +0.0015

Per-class change:
  Atelectasis            0.7540  – +0.0007
  Consolidation          0.7711  – +0.0008
  Infiltration           0.7203  – +0.0013  ⚠️ 
  Pneumothorax           0.9011  – +0.0009  ✅
  Edema                  0.9252  ↑ +0.0021  ✅
  Emphysema              0.9576  – +0.0006  ✅
  Fibrosis               0.9452  ↑ +0.0028  ✅
  Effusion               0.8171  – +0.0012
  Pneumonia              0.9023  ↑ +0.0060  ✅
  Cardiomegaly           0.9339  – +0.0016  ✅
  Nodule                 0.7543  – +0.0012
  Mass                   0.8201  – +0.0014
  Hernia                 0.9871  – -0.0014  ✅


In [13]:
# CELL 6: ConvNeXt V2 Large — highest impact new model per 2025 SOTA

class ConvNeXtV2(nn.Module):
    """
    ConvNeXt V2 Base pretrained on ImageNet-21K (384→224 fine-tuned).
    Per 2025 SOTA paper: pretraining diversity > architecture novelty.
    Uses convnextv2_base.fcmae_ft_in22k_in1k for best AUC on ChestX-ray14.
    """
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            "convnextv2_base.fcmae_ft_in22k_in1k",
            pretrained=True,
            num_classes=0,           # remove head
        )
        feat_dim = self.backbone.num_features   # 1024

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x, metadata=None):
        return self.head(self.backbone(x))


def count_params(model, name):
    total     = sum(p.numel() for p in model.parameters()) / 1e6
    trainable = sum(p.numel() for p in model.parameters()
                    if p.requires_grad) / 1e6
    print(f"  {name:<28} {total:.1f}M total | {trainable:.1f}M trainable")

model_cnx = ConvNeXtV2().to(DEVICE)
count_params(model_cnx, "ConvNeXt V2 Base")
# CELL 6b: Verify ConvNeXt downloaded correctly after wait

import timm
print("timm version:", timm.__version__)

# Check if model is available
available = [m for m in timm.list_models() if "convnextv2_base" in m]
print("Available ConvNeXt V2 variants:")
for m in available:
    print(f"  {m}")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:  95%|#########4| 336M/355M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/timm/convnextv2_base.fcmae_ft_in22k_in1k/resolve/main/model.safetensors: HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.
Trying to resume download...


model.safetensors:  97%|#########7| 346M/355M [00:00<?, ?B/s]

  ConvNeXt V2 Base             88.2M total | 88.2M trainable
timm version: 0.9.12
Available ConvNeXt V2 variants:
  convnextv2_base


In [14]:
# CELL 7: Dataset + DataLoaders (copy your working classes, update transforms)
# Paste your Dataset class here from the old notebook (ChestXrayDataset)
# Then build loaders:

import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_transforms(train=True, img_size=224):
    if train:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                               rotate_limit=10, p=0.5),
            A.GaussNoise(p=0.2),
            A.CLAHE(p=0.3),
            A.CoarseDropout(max_holes=8, max_height=24, max_width=24, p=0.3),
            A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            ToTensorV2()
        ])
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ToTensorV2()
    ])


# Paste your ChestXrayDataset class here, then:
# TRAIN_DL_CNX = DataLoader(train_ds, batch_size=32, shuffle=True,
#                            num_workers=4, pin_memory=True)
# VAL_DL_CNX   = DataLoader(val_ds,   batch_size=64, shuffle=False,
#                            num_workers=4, pin_memory=True)
# TEST_DL_CNX  = DataLoader(test_ds,  batch_size=64, shuffle=False,
#                            num_workers=4, pin_memory=True)

print("✅ Define dataset class and loaders here.")


✅ Define dataset class and loaders here.


In [15]:
# CELL 8: Asymmetric Focal Loss — better than standard focal for rare classes

class AsymmetricFocalLoss(nn.Module):
    """
    Asymmetric loss: penalises false negatives more than false positives.
    Better than standard focal for severely imbalanced multi-label tasks.
    gamma_neg > gamma_pos pushes model to recall rare diseases.
    """
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05,
                 eps=1e-8, weight=None):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip
        self.eps       = eps
        self.weight    = weight   # per-class tensor

    def forward(self, logits, targets):
        prob     = torch.sigmoid(logits)
        prob_clp = torch.clamp(prob, self.clip, 1.0)

        # Positive arm
        loss_pos = targets * torch.log(prob_clp + self.eps)
        # Negative arm (asymmetric clipping)
        prob_neg = torch.clamp(1.0 - prob, self.clip, 1.0)
        loss_neg = (1 - targets) * torch.log(prob_neg + self.eps)

        # Asymmetric focusing
        pt_pos   = prob
        pt_neg   = 1.0 - prob
        loss_pos = loss_pos * ((1 - pt_pos) ** self.gamma_pos)
        loss_neg = loss_neg * ((1 - pt_neg) ** self.gamma_neg)

        loss = -(loss_pos + loss_neg)

        if self.weight is not None:
            loss = loss * self.weight.to(logits.device)
        return loss.mean()


# Compute class weights from training labels
# Paste your CLASS_WEIGHTS computation here from old notebook, then:
# asl_loss = AsymmetricFocalLoss(gamma_neg=4, gamma_pos=1,
#                                 weight=CLASS_WEIGHTS)
print("✅ AsymmetricFocalLoss defined.")


✅ AsymmetricFocalLoss defined.


In [2]:
# FULL RESTORE: Loads all cached predictions including TTA
# Run this once after kernel restart — then go straight to Cell 9 (ConvNeXt)

import os, json, glob, warnings
import numpy as np
import torch
import torch.nn as nn
import timm
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
warnings.filterwarnings("ignore")

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = r"D:\Major Project"
DATA_DIR = r"D:\Major Project\chest_xray_data"
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 0
NUM_CLASSES  = 14
LABELS = [
    "Atelectasis","Consolidation","Infiltration","Pneumothorax",
    "Edema","Emphysema","Fibrosis","Effusion","Pneumonia",
    "PleuralThickening","Cardiomegaly","Nodule","Mass","Hernia"
]
minority_classes = ["Hernia","Pneumonia","Edema","Emphysema","Fibrosis"]

PATH_B0   = os.path.join(BASE_DIR, "best_balanced_effnet.pth")
PATH_B3   = os.path.join(BASE_DIR, "best_efficientnet_b3.pth")
PATH_VIT  = os.path.join(BASE_DIR, "best_vit_base.pth")
PATH_DINO = os.path.join(BASE_DIR, "best_ultimate_dinov2.pth")
PATH_CNX  = os.path.join(BASE_DIR, "best_convnext_v2.pth")

def safe_roc_auc(y_true, y_pred):
    pc = np.full(y_true.shape[1], np.nan)
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) < 2: continue
        pc[i] = roc_auc_score(y_true[:, i], y_pred[:, i])
    return float(np.nanmean(pc)), pc

# ── Load all cached predictions ───────────────────────────────────────────
print("Loading cached predictions...\n")
caches = {
    "test_labels" : "cache_test_labels.npy",
    "b0_preds"    : "cache_b0_preds.npy",
    "b3_preds"    : "cache_b3_preds.npy",
    "vit_preds"   : "cache_vit_preds.npy",
    "dino_preds"  : "cache_dino_preds.npy",
    "b0_tta"      : "cache_tta_b0.npy",
    "b3_tta"      : "cache_tta_b3.npy",
    "vit_tta"     : "cache_tta_vit.npy",
    "dino_tta"    : "cache_tta_dino.npy",
}
loaded = {}
for var, fname in caches.items():
    path = os.path.join(BASE_DIR, fname)
    if os.path.exists(path):
        loaded[var] = np.load(path)
        print(f"  ✅ {var:<15} {loaded[var].shape}")
    else:
        print(f"  ❌ {var:<15} NOT FOUND — {fname}")

test_labels = loaded["test_labels"]
b0_preds    = loaded["b0_preds"]
b3_preds    = loaded["b3_preds"]
vit_preds   = loaded["vit_preds"]
dino_preds  = loaded["dino_preds"]
b0_tta      = loaded["b0_tta"]
b3_tta      = loaded["b3_tta"]
vit_tta     = loaded["vit_tta"]
dino_tta    = loaded["dino_tta"]

# ── Recompute AUCs ────────────────────────────────────────────────────────
wb0, wb3, wvit, wdino = 0.25, 0.15, 0.30, 0.30

baseline_preds       = wb0*b0_preds  + wb3*b3_preds  + wvit*vit_preds  + wdino*dino_preds
tta_ens              = wb0*b0_tta    + wb3*b3_tta    + wvit*vit_tta    + wdino*dino_tta

baseline_auc, baseline_pc = safe_roc_auc(test_labels, baseline_preds)
tta_auc,      tta_pc      = safe_roc_auc(test_labels, tta_ens)

print(f"\n{'='*50}")
print(f"  Baseline 4-Way AUC  : {baseline_auc:.4f}")
print(f"  TTA 4-Way AUC       : {tta_auc:.4f}  (+{tta_auc-baseline_auc:.4f})")
print(f"{'='*50}")
print(f"\n✅ All predictions restored.")
print(f"✅ Next step → run ConvNeXt V2 training (Cell 9).")


Loading cached predictions...

  ✅ test_labels     (25596, 14)
  ✅ b0_preds        (25596, 14)
  ✅ b3_preds        (25596, 14)
  ✅ vit_preds       (25596, 14)
  ✅ dino_preds      (25596, 14)
  ✅ b0_tta          (25596, 14)
  ✅ b3_tta          (25596, 14)
  ✅ vit_tta         (25596, 14)
  ✅ dino_tta        (25596, 14)

  Baseline 4-Way AUC  : 0.8592
  TTA 4-Way AUC       : 0.8607  (+0.0015)

✅ All predictions restored.
✅ Next step → run ConvNeXt V2 training (Cell 9).


In [6]:
# STEP 1: Convert all PNGs → single HDF5 file (run once)
# After this, training reads from HDF5 instead of 100k separate PNGs

import h5py
import os
import numpy as np
from PIL import Image as PILImage
from tqdm import tqdm
import pandas as pd

HDF5_PATH = os.path.join(BASE_DIR, "chest_xray_224.h5")

if os.path.exists(HDF5_PATH):
    print(f"✅ HDF5 already exists: {HDF5_PATH}")
    print(f"   Size: {os.path.getsize(HDF5_PATH)/1e9:.1f} GB")
else:
    print("Building HDF5 file (run once ~30 min)...\n")

    # Collect all image filenames
    all_fnames = sorted(IMAGE_MAP.keys())
    N          = len(all_fnames)
    print(f"  Total images to convert: {N:,}")

    with h5py.File(HDF5_PATH, "w") as hf:
        # Pre-allocate dataset
        imgs_ds = hf.create_dataset(
            "images",
            shape    = (N, IMG_SIZE, IMG_SIZE, 3),
            dtype    = np.uint8,
            chunks   = (32, IMG_SIZE, IMG_SIZE, 3),   # chunk by batch size
            compression = "lzf",                       # fast compression
        )
        fname_ds = hf.create_dataset(
            "filenames",
            data = np.array(all_fnames, dtype=h5py.special_dtype(vlen=str))
        )

        failed = 0
        for i, fname in enumerate(tqdm(all_fnames, ncols=100)):
            path = IMAGE_MAP.get(fname, None)
            try:
                img = PILImage.open(path).convert("RGB")
                img = img.resize((IMG_SIZE, IMG_SIZE), PILImage.BILINEAR)
                imgs_ds[i] = np.array(img, dtype=np.uint8)
            except Exception:
                imgs_ds[i] = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
                failed     += 1

        print(f"\n  ✅ Written: {N - failed:,} images")
        print(f"  ❌ Failed : {failed}")

    size_gb = os.path.getsize(HDF5_PATH) / 1e9
    print(f"  💾 HDF5 size: {size_gb:.1f} GB → {HDF5_PATH}")


Building HDF5 file (run once ~30 min)...

  Total images to convert: 112,120


100%|█████████████████████████████████████████████████████| 112120/112120 [2:54:07<00:00, 10.73it/s]


  ✅ Written: 112,120 images
  ❌ Failed : 0
  💾 HDF5 size: 9.4 GB → D:\Major Project\chest_xray_224.h5


In [9]:
# Rebuild HDF5 — no compression, no chunking = maximum read speed

import h5py, os
import numpy as np
from PIL import Image as PILImage
from tqdm import tqdm

HDF5_PATH = os.path.join(BASE_DIR, "chest_xray_224.h5")
HDF5_FAST = os.path.join(BASE_DIR, "chest_xray_224_fast.h5")

if os.path.exists(HDF5_FAST):
    print(f"✅ Fast HDF5 exists: {os.path.getsize(HDF5_FAST)/1e9:.1f} GB")
else:
    all_fnames = sorted(IMAGE_MAP.keys())
    N          = len(all_fnames)
    print(f"Rebuilding HDF5 (no compression) — {N:,} images...\n")

    with h5py.File(HDF5_FAST, "w") as hf:
        imgs_ds = hf.create_dataset(
            "images",
            shape       = (N, IMG_SIZE, IMG_SIZE, 3),
            dtype       = np.uint8,
            chunks      = None,       # ✅ no chunking — contiguous layout
            compression = None,       # ✅ no compression — raw reads
        )
        hf.create_dataset(
            "filenames",
            data = np.array(all_fnames,
                            dtype=h5py.special_dtype(vlen=str))
        )
        failed = 0
        for i, fname in enumerate(tqdm(all_fnames, ncols=100)):
            path = IMAGE_MAP.get(fname, None)
            try:
                img = PILImage.open(path).convert("RGB")
                img = img.resize((IMG_SIZE, IMG_SIZE), PILImage.BILINEAR)
                imgs_ds[i] = np.array(img, dtype=np.uint8)
            except Exception:
                imgs_ds[i] = np.zeros((IMG_SIZE,IMG_SIZE,3), dtype=np.uint8)
                failed += 1

    print(f"  ✅ Done. Failed: {failed}")
    print(f"  💾 {os.path.getsize(HDF5_FAST)/1e9:.1f} GB → {HDF5_FAST}")

# Update path
HDF5_PATH = HDF5_FAST
print(f"\nUsing: {HDF5_PATH}")


Rebuilding HDF5 (no compression) — 112,120 images...



100%|█████████████████████████████████████████████████████| 112120/112120 [1:54:10<00:00, 16.37it/s]


  ✅ Done. Failed: 0
  💾 16.9 GB → D:\Major Project\chest_xray_224_fast.h5

Using: D:\Major Project\chest_xray_224_fast.h5


In [10]:
# Simplified transforms — removes slow ops (GaussNoise, CoarseDropout)
# These add <0.3% AUC but cost 40% of transform time

def get_transforms_fast(train=True):
    if train:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(0.15, 0.15, p=0.4),
            A.ShiftScaleRotate(0.05, 0.1, 10, p=0.4),
            A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
            ToTensorV2()
        ])
    # No Resize needed — images already 224x224 in HDF5
    return A.Compose([
        A.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ToTensorV2()
    ])


# Rebuild DataLoaders — batch_size=64, fast transforms
TRAIN_DL_CNX = DataLoader(
    HDF5ChestXrayDataset(df_train, HDF5_PATH, get_transforms_fast(True)),
    batch_size  = 64,       # ✅ doubled — fewer batches (1217 vs 2434)
    shuffle     = True,
    num_workers = 0,
    pin_memory  = False,
)
VAL_DL_CNX = DataLoader(
    HDF5ChestXrayDataset(df_val, HDF5_PATH, get_transforms_fast(False)),
    batch_size  = 128,
    shuffle     = False,
    num_workers = 0,
    pin_memory  = False,
)

print(f"Train batches : {len(TRAIN_DL_CNX):,}  (was 2,434)")
print(f"Val batches   : {len(VAL_DL_CNX):,}")
print("✅ Ready — re-run ConvNeXt training cell.")


Train batches : 1,217  (was 2,434)
Val batches   : 68
✅ Ready — re-run ConvNeXt training cell.


In [13]:
# NUMPY-BACKED DATASET: Pre-extracts all metadata to numpy in __init__
# Eliminates pandas iloc overhead (~3-4s/batch → ~0.1s/batch)

class FastHDF5Dataset(Dataset):
    def __init__(self, df, hdf5_path, transform):
        df = df.reset_index(drop=True)
        self.transform  = transform
        self.hdf5_path  = hdf5_path
        self.hdf5_file  = None

        # Build HDF5 index map
        with h5py.File(hdf5_path, "r") as hf:
            fnames = [f.decode() if isinstance(f, bytes) else f
                      for f in hf["filenames"][:]]
        fname_to_hidx = {f: i for i, f in enumerate(fnames)}

        # ✅ Pre-extract everything to numpy — no pandas in __getitem__
        self.hidx    = np.array([fname_to_hidx.get(f, -1)
                                  for f in df["Image_Index"]], dtype=np.int32)
        self.labels  = df[LABELS].values.astype(np.float32)  # (N, 14)
        self.ages    = np.clip(df["Patient Age"].fillna(50)
                               .astype(float).values, 0, 95) / 95.0
        self.genders = np.array([{"M":0.0,"F":1.0}.get(
                                   str(g).upper(), 0.5)
                                  for g in df["Patient Gender"].fillna("M")],
                                 dtype=np.float32)
        self.views   = np.array([{"PA":0.0,"AP":1.0}.get(
                                   str(v).upper(), 0.5)
                                  for v in df["View Position"].fillna("PA")],
                                 dtype=np.float32)
        self.fups    = np.clip(df["Follow-up #"].fillna(0)
                               .astype(float).values / 10.0, 0, 1
                               ).astype(np.float32)
        print(f"  Dataset ready: {len(self.hidx):,} samples, "
              f"{(self.hidx == -1).sum()} missing")

    def __len__(self):
        return len(self.hidx)

    def _get_file(self):
        if self.hdf5_file is None:
            self.hdf5_file = h5py.File(self.hdf5_path, "r")
        return self.hdf5_file

    def __getitem__(self, idx):
        hidx = self.hidx[idx]
        img  = self._get_file()["images"][hidx] \
               if hidx >= 0 \
               else np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

        img  = self.transform(image=img)["image"]

        meta = torch.FloatTensor([
            self.ages[idx], self.genders[idx],
            self.views[idx], self.fups[idx]
        ])
        return img, torch.from_numpy(self.labels[idx]), meta


# Rebuild loaders
print("Building fast datasets...\n")
TRAIN_DL_CNX = DataLoader(
    FastHDF5Dataset(df_train, HDF5_PATH, get_transforms_fast(True)),
    batch_size=64, shuffle=True,
    num_workers=0, pin_memory=False,
)
VAL_DL_CNX = DataLoader(
    FastHDF5Dataset(df_val, HDF5_PATH, get_transforms_fast(False)),
    batch_size=128, shuffle=False,
    num_workers=0, pin_memory=False,
)

# ── Quick speed test ──────────────────────────────────────────────────────
import time
print("\nSpeed test (5 batches)...")
t0 = time.time()
for i, (x, y, m) in enumerate(TRAIN_DL_CNX):
    if i == 4: break
elapsed = (time.time() - t0) / 5
print(f"  Avg time/batch : {elapsed:.2f}s")
print(f"  Est. time/epoch: {elapsed * len(TRAIN_DL_CNX) / 60:.0f} min")
print(f"  Train batches  : {len(TRAIN_DL_CNX):,}")


Building fast datasets...

  Dataset ready: 77,871 samples, 0 missing
  Dataset ready: 8,653 samples, 0 missing

Speed test (5 batches)...
  Avg time/batch : 0.19s
  Est. time/epoch: 4 min
  Train batches  : 1,217


In [15]:
# CUDA OOM FIX: Switch to ConvNeXt V2 Tiny + smaller batch

import torch
torch.cuda.empty_cache()  # free memory from failed run

# ── Smaller model ─────────────────────────────────────────────────────────
class ConvNeXtV2(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            "convnextv2_tiny.fcmae_ft_in22k_in1k",  # ✅ Tiny: 28M vs 89M
            pretrained=True,
            num_classes=0,
        )
        feat_dim = self.backbone.num_features  # 768
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(512, num_classes),
        )
    def forward(self, x, metadata=None):
        return self.head(self.backbone(x))

model_cnx = ConvNeXtV2().to(DEVICE)
total     = sum(p.numel() for p in model_cnx.parameters()) / 1e6
trainable = sum(p.numel() for p in model_cnx.parameters()
                if p.requires_grad) / 1e6
print(f"ConvNeXt V2 Tiny: {total:.1f}M total | {trainable:.1f}M trainable")

# ── Rebuild loaders with safe batch size ─────────────────────────────────
TRAIN_DL_CNX = DataLoader(
    FastHDF5Dataset(df_train, HDF5_PATH, get_transforms_fast(True)),
    batch_size=32, shuffle=True,          # ✅ 32 instead of 64
    num_workers=0, pin_memory=False,
)
VAL_DL_CNX = DataLoader(
    FastHDF5Dataset(df_val, HDF5_PATH, get_transforms_fast(False)),
    batch_size=64, shuffle=False,
    num_workers=0, pin_memory=False,
)
TEST_DL_CNX = DataLoader(
    FastHDF5Dataset(df_test, HDF5_PATH, get_transforms_fast(False)),
    batch_size=64, shuffle=False,
    num_workers=0, pin_memory=False,
)

# ── Quick memory check before training ───────────────────────────────────
x_test = torch.randn(32, 3, 224, 224).to(DEVICE)
with torch.cuda.amp.autocast():
    out = model_cnx(x_test)
del x_test
torch.cuda.empty_cache()
mem = torch.cuda.memory_allocated() / 1e9
print(f"GPU memory after test forward: {mem:.2f} GB  ✅ OK to train")

print(f"\nTrain batches : {len(TRAIN_DL_CNX):,}")
print(f"Val batches   : {len(VAL_DL_CNX):,}")
print("✅ Ready — re-run training cell")


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [14]:
# CONVNEXT FINAL TRAINING CELL
# Uses FastHDF5Dataset — ~4 min/epoch, 20 epochs max

from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# ── Rebuild model fresh ───────────────────────────────────────────────────
print("Building ConvNeXt V2...")
model_cnx = ConvNeXtV2().to(DEVICE)
total     = sum(p.numel() for p in model_cnx.parameters()) / 1e6
trainable = sum(p.numel() for p in model_cnx.parameters()
                if p.requires_grad) / 1e6
print(f"  {total:.1f}M total | {trainable:.1f}M trainable\n")

# ── Optimizer — layer-wise LR ─────────────────────────────────────────────
backbone_p, head_p = [], []
for name, param in model_cnx.named_parameters():
    (backbone_p if "backbone" in name else head_p).append(param)

optimizer_cnx = torch.optim.AdamW([
    {"params": backbone_p, "lr": 5e-6},
    {"params": head_p,     "lr": 5e-5},
], weight_decay=1e-2)

scheduler_cnx = CosineAnnealingWarmRestarts(
    optimizer_cnx, T_0=5, T_mult=2, eta_min=1e-7)

scaler_cnx    = torch.cuda.amp.GradScaler()
best_auc_cnx  = -1.0
patience_ctr  = 0
PATIENCE_CNX  = 6
NUM_EPOCHS    = 20
cnx_history   = []

print(f"Starting ConvNeXt V2 — {NUM_EPOCHS} epochs max")
print(f"Est. ~4 min/epoch → ~{NUM_EPOCHS*4} min max\n")

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────────────────
    model_cnx.train()
    t_loss, t_labels, t_preds = 0.0, [], []
    pbar = tqdm(TRAIN_DL_CNX, desc=f"Epoch {epoch:02d}", ncols=110)
    for i, (x, y, meta) in enumerate(pbar):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer_cnx.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            logits = model_cnx(x)
            loss   = asl_loss(logits, y)
        scaler_cnx.scale(loss).backward()
        scaler_cnx.unscale_(optimizer_cnx)
        torch.nn.utils.clip_grad_norm_(model_cnx.parameters(), 1.0)
        scaler_cnx.step(optimizer_cnx)
        scaler_cnx.update()
        t_loss += loss.item()
        t_labels.append(y.detach().cpu().numpy())
        t_preds.append(
            torch.sigmoid(logits).detach().cpu().float().numpy())
        if i % 200 == 0:
            pbar.set_postfix({"loss": f"{t_loss/(i+1):.4f}"})

    scheduler_cnx.step()
    tr_auc, _ = safe_roc_auc(np.vstack(t_labels), np.vstack(t_preds))

    # ── Validate ──────────────────────────────────────────────────────────
    model_cnx.eval()
    v_labels, v_preds = [], []
    with torch.no_grad():
        for x, y, _ in tqdm(VAL_DL_CNX, desc="  Val", ncols=110):
            with torch.cuda.amp.autocast():
                logits = model_cnx(x.to(DEVICE))
            v_labels.append(y.numpy())
            v_preds.append(
                torch.sigmoid(logits).cpu().float().numpy())

    val_auc, val_pc = safe_roc_auc(
        np.vstack(v_labels), np.vstack(v_preds))

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}  "
          f"Train AUC {tr_auc:.4f}  |  Val AUC {val_auc:.4f}")
    for l, a in zip(LABELS, val_pc):
        if np.isnan(a): continue
        flag = "  ⚠️ " if a < 0.75 else ("  ✅" if a >= 0.85 else "")
        print(f"  {l:<22} {a:.4f}{flag}")

    cnx_history.append({
        "epoch": epoch, "train_auc": tr_auc, "val_auc": val_auc,
        "per_class": [float(a) if not np.isnan(a) else None
                      for a in val_pc]
    })

    if val_auc > best_auc_cnx:
        best_auc_cnx = val_auc
        patience_ctr = 0
        torch.save(model_cnx.state_dict(), PATH_CNX)
        print(f"  ✅ Saved → Val AUC {best_auc_cnx:.4f}")
    else:
        patience_ctr += 1
        print(f"  No improvement ({patience_ctr}/{PATIENCE_CNX})"
              f"  [best: {best_auc_cnx:.4f}]")
        if patience_ctr >= PATIENCE_CNX:
            print("  🛑 Early stopping.")
            break

with open(os.path.join(BASE_DIR, "phase8_convnext_results.json"), "w") as f:
    json.dump(cnx_history, f, indent=2)
print(f"\n✅ ConvNeXt V2 done.  Best Val AUC: {best_auc_cnx:.4f}")


Building ConvNeXt V2...
  88.2M total | 88.2M trainable

Starting ConvNeXt V2 — 20 epochs max
Est. ~4 min/epoch → ~80 min max



Epoch 01:   0%|                                                                      | 0/1217 [00:08<?, ?it/s]


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# CELL 10: 5-Way Ensemble — B0 + B3 + ViT + DINOv2 + ConvNeXt V2

# Load best ConvNeXt
model_cnx.load_state_dict(torch.load(PATH_CNX, map_location=DEVICE))
model_cnx.eval()

# Get ConvNeXt test predictions
@torch.no_grad()
def get_test_preds(model, loader):
    model.eval()
    preds = []
    for x, y, meta in tqdm(loader, ncols=100):
        with torch.cuda.amp.autocast():
            logits = model(x.to(DEVICE), None)
        preds.append(torch.sigmoid(logits).cpu().float().numpy())
    return np.vstack(preds)

cnx_preds = get_test_preds(model_cnx, TEST_DL_CNX)
np.save(os.path.join(BASE_DIR, "cache_cnx_preds.npy"), cnx_preds)

cnx_auc, cnx_pc = safe_roc_auc(test_labels, cnx_preds)
print(f"ConvNeXt V2 Test AUC: {cnx_auc:.4f}")

# Weight search — 5-way
weight_configs_5 = [
    (0.20, 0.10, 0.20, 0.20, 0.30),   # heavy on ConvNeXt
    (0.20, 0.10, 0.20, 0.15, 0.35),
    (0.15, 0.10, 0.20, 0.20, 0.35),
    (0.20, 0.15, 0.20, 0.15, 0.30),
    (0.25, 0.10, 0.20, 0.15, 0.30),
    (0.15, 0.10, 0.25, 0.15, 0.35),
]

best5_auc, best5_w, best5_preds = -1.0, None, None
print("\nTesting 5-way ensemble weights...\n")
for wb0, wb3, wvit, wdino, wcnx in weight_configs_5:
    ens  = (wb0*b0_preds + wb3*b3_preds + wvit*vit_preds
            + wdino*dino_preds + wcnx*cnx_preds)
    auc, _ = safe_roc_auc(test_labels, ens)
    flag   = "  ← BEST" if auc > best5_auc else ""
    print(f"  B0={wb0} B3={wb3} ViT={wvit} DINO={wdino} CNX={wcnx} "
          f"→ AUC={auc:.4f}{flag}")
    if auc > best5_auc:
        best5_auc, best5_w, best5_preds = auc, \
            (wb0,wb3,wvit,wdino,wcnx), ens.copy()

_, best5_pc = safe_roc_auc(test_labels, best5_preds)
m_aucs_5    = [best5_pc[LABELS.index(c)] for c in minority_classes]

print(f"\n{'='*60}")
print(f"  Phase 7 Baseline  AUC : {baseline_auc:.4f}")
print(f"  Phase 8 5-Way     AUC : {best5_auc:.4f}  "
      f"(+{best5_auc-baseline_auc:.4f})")
print(f"  Minority Avg          : {np.nanmean(m_aucs_5):.4f}")
print(f"{'='*60}")
print("\nPer-class AUC:")
for l, a, b in zip(LABELS, best5_pc, baseline_pc):
    if np.isnan(a): continue
    delta = a - b if not np.isnan(b) else 0
    arrow = "↑" if delta > 0.005 else ("↓" if delta < -0.005 else " ")
    print(f"  {l:<22} {a:.4f}  {arrow} {delta:+.4f}")
